# Notebook 04: Model Training — `kinkySobel`

### English
This notebook trains a **64×64 conditional DCGAN** for two classes: dog and cat. The Generator receives a noise vector and species label. The Discriminator receives the RGB image, the species-conditioning channel, and a Sobel edge channel.

Training is split into three stages:

| Stage | Epochs | Main purpose | Main methods |
|---|---:|---|---|
| `stage1` | 150 | Learn basic image structure | Equal G/D learning rates |
| `stage2` | 200 | Refine while controlling D | TTUR, instance noise, DiffAugment, EMA |
| `stage3` | up to 100 | Improve sharpness and preserve diversity | Low LR, DiffAugment, EMA, mode-seeking loss, collapse early stop |

All stages use the same 64×64 architecture. Stage 2 continues from Stage 1; Stage 3 starts from the Stage 2 EMA Generator. Model files are stored under `models/kinkySobel/`.

### Tiếng Việt
Notebook này train một **conditional DCGAN 64×64** cho hai lớp chó và mèo. Generator nhận vector noise và nhãn loài. Discriminator nhận ảnh RGB, kênh conditioning theo loài và kênh biên Sobel.

Quá trình train gồm ba stage:

| Stage | Epoch | Mục tiêu chính | Phương pháp chính |
|---|---:|---|---|
| `stage1` | 150 | Học cấu trúc ảnh cơ bản | LR G/D bằng nhau |
| `stage2` | 200 | Tinh chỉnh và kiểm soát D | TTUR, instance noise, DiffAugment, EMA |
| `stage3` | tối đa 100 | Tăng độ nét và giữ đa dạng | LR thấp, DiffAugment, EMA, mode-seeking loss, early stop khi collapse |

Ba stage dùng cùng kiến trúc 64×64. Stage 2 tiếp tục từ Stage 1; Stage 3 bắt đầu từ EMA Generator của Stage 2. Model được lưu trong `models/kinkySobel/`.


## 1. Environment Setup

### English
Import TensorFlow/Keras, check GPU availability, and set `RANDOM_SEED = 42` for reproducible random initialization and sampling.

### Tiếng Việt
Import TensorFlow/Keras, kiểm tra GPU và đặt `RANDOM_SEED = 42` để việc khởi tạo và lấy mẫu ngẫu nhiên có thể tái lập.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import random

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU(s) available: {[g.name for g in gpus]}")
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> GPU before training.")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

## 2. Mount Google Drive and Locate the Project

### English
Mount Google Drive and define project paths. Weights, training history, checkpoints, and sample figures are saved inside `MyDrive/dog-gan-project`.

### Tiếng Việt
Mount Google Drive và khai báo các đường dẫn của project. Weight, lịch sử train, checkpoint và ảnh mẫu được lưu trong `MyDrive/dog-gan-project`.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/dog-gan-project")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "04_training"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PROCESSED.exists(), f"Processed data not found at {DATA_PROCESSED} -- run 03_preprocessing.ipynb and upload its output first."
print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {DATA_PROCESSED}")

## 3. Load Manifests and Shared Hyperparameters

### English
Load the 64×64 train/validation manifests and define parameters shared by all stages.

Key values:
- `IMAGE_SIZE = 64`: training resolution.
- `NOISE_DIM = 100`: Generator latent-vector size.
- `EMBEDDING_DIM = 50`: species-label embedding size.
- `BATCH_SIZE = 128`: images per training batch.
- `CHECKPOINT_EVERY = 5`: save rolling checkpoints and sample grids every 5 epochs.
- `BETA_1 = 0.5`: Adam momentum parameter used by both optimizers.
- `REAL_LABEL_SMOOTHING = 0.9`: real targets use 0.9 instead of 1.0.
- `COLLAPSE_STOP_RATIO = 0.60`, `COLLAPSE_PATIENCE = 2`: Stage 3 collapse guard.

### Tiếng Việt
Đọc manifest train/validation 64×64 và khai báo các tham số dùng chung cho cả ba stage.

Các giá trị chính:
- `IMAGE_SIZE = 64`: độ phân giải train.
- `NOISE_DIM = 100`: kích thước latent vector của Generator.
- `EMBEDDING_DIM = 50`: kích thước embedding của nhãn loài.
- `BATCH_SIZE = 128`: số ảnh trong một batch.
- `CHECKPOINT_EVERY = 5`: lưu checkpoint quay vòng và ảnh mẫu mỗi 5 epoch.
- `BETA_1 = 0.5`: tham số momentum của Adam cho cả G và D.
- `REAL_LABEL_SMOOTHING = 0.9`: nhãn ảnh thật dùng 0.9 thay vì 1.0.
- `COLLAPSE_STOP_RATIO = 0.60`, `COLLAPSE_PATIENCE = 2`: điều kiện bảo vệ Stage 3 khỏi collapse.


### 64×64 Dataset Version Safety

### English
This notebook reads `train_manifest_64.csv` and `val_manifest_64.csv`, which point to `images_64/`. The previous 128×128 dataset and manifests are not overwritten.

### Tiếng Việt
Notebook chỉ đọc `train_manifest_64.csv` và `val_manifest_64.csv`, trỏ tới `images_64/`. Dữ liệu và manifest 128×128 cũ không bị ghi đè.


In [ ]:
import pandas as pd

train_manifest = pd.read_csv(DATA_PROCESSED / "train_manifest_64.csv")
val_manifest = pd.read_csv(DATA_PROCESSED / "val_manifest_64.csv")

SPECIES_TO_LABEL = {"dog": 0, "cat": 1}
LABEL_TO_SPECIES = {v: k for k, v in SPECIES_TO_LABEL.items()}
NUM_CLASSES = len(SPECIES_TO_LABEL)

IMAGE_SIZE = 64
NOISE_DIM = 100
EMBEDDING_DIM = 50
BATCH_SIZE = 128
CHECKPOINT_EVERY = 5
MODEL_SUBDIR = "kinkySobel"

BETA_1 = 0.5
REAL_LABEL_SMOOTHING = 0.9  # one-sided label smoothing on the "real" target

# Collapse guard (used by stage 3's early stop and by the warnings in every stage)
HEALTH_TARGET = 0.85        # struct diversity we would like to hold, as a fraction of the real-image reference
COLLAPSE_STOP_RATIO = 0.60  # below this fraction of the real reference = treat as collapsing
COLLAPSE_PATIENCE = 2       # consecutive checkpoint evaluations below the ratio before stage 3 stops

print(f"Train images: {len(train_manifest)} -- {train_manifest['species'].value_counts().to_dict()}")
print(f"Val images  : {len(val_manifest)} -- {val_manifest['species'].value_counts().to_dict()}")
print(f"Species -> label mapping: {SPECIES_TO_LABEL}")
print(f"Steps per epoch (approx): {len(train_manifest) // BATCH_SIZE}")

### 3b. Three-Stage Training Configuration

### English
The `STAGES` dictionary contains every parameter that changes between stages.

| Parameter | Stage 1 | Stage 2 | Stage 3 |
|---|---:|---:|---:|
| Epochs | 150 | 200 | 100 max |
| G LR | `2e-4` | `1.5e-4` | `5e-5` |
| D LR | `2e-4` | `5e-5` | `2.5e-5` |
| Instance noise | 0 | 0.05 → 0 | 0 |
| DiffAugment | off | translation | translation |
| EMA decay | off | 0.95 | 0.90 |
| Mode-seeking weight | 0 | 0 | 0.2 |
| LR decay starts | none | 70% | 30% |
| LR floor | 1.0 | 0.4 | 0.3 |
| Snapshot interval | 25 | 25 | 10 |
| Collapse early stop | off | off | on |

`lr_floor` is a fraction of the stage's base LR. Instance noise in Stage 2 is linearly reduced to zero during the first 60% of the stage.

### Tiếng Việt
Dictionary `STAGES` chứa toàn bộ tham số thay đổi giữa các stage.

| Tham số | Stage 1 | Stage 2 | Stage 3 |
|---|---:|---:|---:|
| Epoch | 150 | 200 | tối đa 100 |
| LR G | `2e-4` | `1.5e-4` | `5e-5` |
| LR D | `2e-4` | `5e-5` | `2.5e-5` |
| Instance noise | 0 | 0.05 → 0 | 0 |
| DiffAugment | tắt | translation | translation |
| EMA decay | tắt | 0.95 | 0.90 |
| Mode-seeking weight | 0 | 0 | 0.2 |
| Bắt đầu giảm LR | không giảm | 70% | 30% |
| Sàn LR | 1.0 | 0.4 | 0.3 |
| Snapshot | mỗi 25 epoch | mỗi 25 epoch | mỗi 10 epoch |
| Early stop collapse | tắt | tắt | bật |

`lr_floor` là tỷ lệ so với LR gốc của stage. Instance noise ở Stage 2 giảm tuyến tính về 0 trong 60% đầu stage.


### Web Progression Snapshots

### English
Extra Generator weights are saved for the future GitHub web animation. The same latent vector `z` and species label can later be passed through these checkpoints to show one generated subject evolving from an early uncanny state to the final image.

Saved progression points:
- Stage 1: `1, 3, 5, 10, 15, 25, 50, 100, 150`
- Stage 2: `25, 100, 200`
- Stage 3: `10, 50, 100` when reached
- Final: the best Stage 3 Generator is copied as `generator_final_best.weights.h5`

Stage 1 stores the normal Generator; Stage 2/3 store the EMA Generator used for evaluation. Files are isolated in `models/kinkySobel/web_progression/` and do not affect resume checkpoints.

### Tiếng Việt
Notebook lưu thêm weight Generator để dùng cho animation trên web GitHub sau này. Khi deploy, cùng một latent vector `z` và nhãn loài sẽ được chạy qua các checkpoint này để cho thấy cùng một ảnh chuyển dần từ trạng thái uncanny ban đầu đến ảnh hoàn chỉnh.

Các mốc được lưu:
- Stage 1: `1, 3, 5, 10, 15, 25, 50, 100, 150`
- Stage 2: `25, 100, 200`
- Stage 3: `10, 50, 100` nếu train tới các mốc đó
- Cuối cùng: best Generator của Stage 3 được copy thành `generator_final_best.weights.h5`

Stage 1 lưu Generator thường; Stage 2/3 lưu EMA Generator đang dùng để đánh giá. Các file nằm riêng trong `models/kinkySobel/web_progression/` và không ảnh hưởng checkpoint dùng để resume.


In [ ]:
STAGES = {
    "stage1": dict(
        epochs=150,
        gen_lr=2e-4, disc_lr=2e-4,      # equal LR -- the proven fast-structure recipe
        instance_noise=0.0, diffaug="", ema_decay=0.0, ms_weight=0.0,
        lr_decay_from=1.0, lr_floor=1.0,  # constant LR for the whole stage
        init_from=None, init_from_ema=False,
        snapshot_every=25, early_stop=False,
    ),
    "stage2": dict(
        epochs=200,  # a long, safe refinement runway under TTUR
        gen_lr=1.5e-4, disc_lr=5e-5,    # TTUR: D deliberately slowed down
        instance_noise=0.05, diffaug="translation", ema_decay=0.95, ms_weight=0.0,
        lr_decay_from=0.7, lr_floor=0.4,  # decay starts at epoch 140, floor reached at epoch 200
        init_from="stage1", init_from_ema=False,
        snapshot_every=25, early_stop=False,
    ),
    "stage3": dict(
        epochs=100,  # enough runway for a sharpening pass to do real work
        gen_lr=5e-5, disc_lr=2.5e-5,    # very low: polish, do not re-learn
        instance_noise=0.0,             # off on purpose -- noise protects blur, we want sharpness
        diffaug="translation", ema_decay=0.90,
        ms_weight=0.2,   # moderate: pushes back on collapse without overpowering sharpening
        lr_decay_from=0.3, lr_floor=0.3,  # keeps some learning signal alive through the end of the stage
        init_from="stage2", init_from_ema=True,
        snapshot_every=10, early_stop=True,
    ),
}

for key, cfg in STAGES.items():
    print(f"{key}: {cfg['epochs']} epochs, G lr={cfg['gen_lr']:.1e}, D lr={cfg['disc_lr']:.1e}, "
          f"noise={cfg['instance_noise']}, diffaug='{cfg['diffaug']}', ema={cfg['ema_decay']}, "
          f"ms={cfg['ms_weight']}, lr_floor={cfg['lr_floor']}, early_stop={cfg['early_stop']}")


# Extra Generator checkpoints kept specifically for the later web animation.
# They do NOT replace rolling checkpoints, permanent snapshots, or best-model saving.
WEB_PROGRESSION_EPOCHS = {
    "stage1": [1, 3, 5, 10, 15, 25, 50, 100, 150],
    "stage2": [25, 100, 200],
    "stage3": [10, 50, 100],
}
WEB_PROGRESSION_DIR = MODELS_DIR / MODEL_SUBDIR / "web_progression"
WEB_PROGRESSION_MANIFEST = WEB_PROGRESSION_DIR / "manifest.json"

print("Web progression checkpoints:")
for stage_key, epochs in WEB_PROGRESSION_EPOCHS.items():
    print(f"  {stage_key}: {epochs}")


## 4. Build the `tf.data` Input Pipeline

### English
Copy processed images to local Colab storage for faster reading, decode them, resize/confirm them as 64×64 RGB, normalize pixels to `[-1, 1]`, attach species labels, then batch and prefetch the data.

### Tiếng Việt
Copy ảnh đã xử lý sang ổ local của Colab để đọc nhanh hơn, decode ảnh, đảm bảo RGB 64×64, chuẩn hóa pixel về `[-1, 1]`, gắn nhãn loài, sau đó batch và prefetch dữ liệu.


In [ ]:
import shutil

LOCAL_IMAGES_DIR = Path("/content/data_local/images_64")

if not LOCAL_IMAGES_DIR.exists():
    LOCAL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    print("Copying images_64 to local disk (first time this session)...")
    shutil.copytree(DATA_PROCESSED / "images_64", LOCAL_IMAGES_DIR, dirs_exist_ok=True)
    print(f"Copied {len(list(LOCAL_IMAGES_DIR.glob('*')))} files.")
else:
    print("Local copy of images_64 already present, skipping copy.")

In [ ]:
def load_image(filepath, label):
    raw = tf.io.read_file(filepath)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method="area")
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0  # normalize to [-1, 1]
    return image, label


def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    return image, label


def build_dataset(manifest_df, images_dir, batch_size, shuffle=True, augment_flag=True):
    filenames = [Path(fp.replace("\\", "/")).name for fp in manifest_df["filepath"]]
    filepaths = [str(images_dir / name) for name in filenames]
    labels = manifest_df["species"].map(SPECIES_TO_LABEL).astype("int32").to_numpy()

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_flag:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = build_dataset(train_manifest, LOCAL_IMAGES_DIR, BATCH_SIZE, shuffle=True, augment_flag=True)
val_ds = build_dataset(val_manifest, LOCAL_IMAGES_DIR, BATCH_SIZE, shuffle=False, augment_flag=False)

print(f"train_ds element spec: {train_ds.element_spec}")

In [ ]:
import matplotlib.pyplot as plt

sample_images, sample_labels = next(iter(train_ds))
sample_images_disp = ((sample_images.numpy() + 1.0) / 2.0).clip(0, 1)

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(sample_images_disp[i])
    ax.set_title(LABEL_TO_SPECIES[int(sample_labels[i])], fontsize=9)
    ax.axis("off")
fig.suptitle("Real training batch (after normalization and flip augmentation)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_real_batch.png", dpi=150)
plt.show()

## 5. Sobel Auxiliary Channel

### English
Compute a grayscale Sobel edge-magnitude map for each image and normalize it to `[-1, 1]`. This extra channel gives the Discriminator explicit information about edges and local structure.

### Tiếng Việt
Tính bản đồ độ lớn cạnh Sobel từ ảnh grayscale và chuẩn hóa về `[-1, 1]`. Kênh phụ này cung cấp cho Discriminator thông tin trực tiếp về biên và cấu trúc cục bộ.


In [ ]:
def _per_image_normalize(magnitude):
    """Min-max normalize a (B, H, W, 1) magnitude map to [-1, 1], per image."""
    min_val = tf.reduce_min(magnitude, axis=[1, 2, 3], keepdims=True)
    max_val = tf.reduce_max(magnitude, axis=[1, 2, 3], keepdims=True)
    normalized = (magnitude - min_val) / (max_val - min_val + 1e-8)
    return normalized * 2.0 - 1.0


def sobel_channel(images):
    """images: (B, H, W, 3) float32 in [-1, 1]. Returns (B, H, W, 1) float32 in [-1, 1]."""
    gray = tf.image.rgb_to_grayscale((images + 1.0) / 2.0)
    edges = tf.image.sobel_edges(gray)  # (B, H, W, 1, 2) -- [dy, dx]
    magnitude = tf.sqrt(tf.reduce_sum(tf.square(edges), axis=-1) + 1e-8)  # (B, H, W, 1)
    return _per_image_normalize(magnitude)


sobel_demo = sobel_channel(sample_images).numpy()

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for col in range(6):
    axes[0, col].imshow(sample_images_disp[col])
    axes[0, col].set_title("RGB", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(sobel_demo[col, :, :, 0], cmap="gray")
    axes[1, col].set_title("Sobel channel", fontsize=9)
    axes[1, col].axis("off")
fig.suptitle("Discriminator auxiliary channel (Sobel)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_auxiliary_channel.png", dpi=150)
plt.show()

## 6. Species-Conditioning Channel

### English
Convert each dog/cat label into a constant spatial channel. The Discriminator receives this channel together with the image, so its real/fake decision is conditioned on the requested species.

### Tiếng Việt
Chuyển nhãn chó/mèo thành một kênh không gian có giá trị cố định. Discriminator nhận kênh này cùng với ảnh để quyết định thật/giả theo đúng loài được yêu cầu.


In [ ]:
def species_channel(labels, size=IMAGE_SIZE):
    """labels: (B,) int32 in {0, 1}. Returns (B, size, size, 1) float32: -1 for dog, +1 for cat."""
    value = tf.cast(labels, tf.float32) * 2.0 - 1.0
    value = tf.reshape(value, [-1, 1, 1, 1])
    batch_size = tf.shape(labels)[0]
    return tf.ones([batch_size, size, size, 1], dtype=tf.float32) * value


species_demo = species_channel(sample_labels).numpy()
dog_idx = int(np.argmax(sample_labels.numpy() == 0))
cat_idx = int(np.argmax(sample_labels.numpy() == 1))

fig, axes = plt.subplots(1, 2, figsize=(6, 3.2))
axes[0].imshow(species_demo[dog_idx, :, :, 0], cmap="gray", vmin=-1, vmax=1)
axes[0].set_title(f"Species channel -- dog (value={species_demo[dog_idx,0,0,0]:.0f})", fontsize=9)
axes[0].axis("off")
axes[1].imshow(species_demo[cat_idx, :, :, 0], cmap="gray", vmin=-1, vmax=1)
axes[1].set_title(f"Species channel -- cat (value={species_demo[cat_idx,0,0,0]:.0f})", fontsize=9)
axes[1].axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_species_channel.png", dpi=150)
plt.show()

## 7. Generator Architecture

### English
The Generator takes:
- a 100-D noise vector `z`;
- a species label embedded into 50 dimensions.

They are concatenated, projected to `8×8×256`, then upsampled with transposed convolutions:

`8×8×256 → 16×16×128 → 32×32×64 → 64×64×3`

Batch Normalization and LeakyReLU are used in hidden layers. The output uses `tanh`, matching image normalization to `[-1, 1]`.

### Tiếng Việt
Generator nhận:
- noise vector 100 chiều `z`;
- nhãn loài được embedding thành 50 chiều.

Hai đầu vào được nối lại, chiếu thành `8×8×256`, rồi upsample bằng transposed convolution:

`8×8×256 → 16×16×128 → 32×32×64 → 64×64×3`

Các tầng ẩn dùng Batch Normalization và LeakyReLU. Output dùng `tanh`, phù hợp với dữ liệu ảnh đã chuẩn hóa về `[-1, 1]`.


In [ ]:
from tensorflow.keras import layers, Model


def build_generator(noise_dim=NOISE_DIM, num_classes=NUM_CLASSES, embedding_dim=EMBEDDING_DIM, name="generator"):
    noise_input = layers.Input(shape=(noise_dim,), name="noise")
    label_input = layers.Input(shape=(), dtype="int32", name="label")

    label_embed = layers.Embedding(num_classes, embedding_dim)(label_input)
    x = layers.Concatenate()([noise_input, label_embed])

    x = layers.Dense(8 * 8 * 256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((8, 8, 256))(x)

    x = layers.Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False)(x)  # 16x16
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False)(x)  # 32x32
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    output = layers.Conv2DTranspose(3, 4, strides=2, padding="same", activation="tanh")(x)  # 64x64

    return Model([noise_input, label_input], output, name=name)


_g = build_generator()
_g.summary()
print(f"\nTotal Generator parameters: {_g.count_params():,}")
del _g

## 8. Discriminator Architecture

### English
The Discriminator concatenates five input channels:
- 3 RGB channels;
- 1 species-conditioning channel;
- 1 Sobel edge channel.

The spatial path is:

`64×64×5 → 32×32×64 → 16×16×128 → 8×8×256 → logit`

LeakyReLU is used throughout; Dropout is applied in the first two convolution blocks; Batch Normalization is used after the second and third convolutions.

### Tiếng Việt
Discriminator ghép năm kênh đầu vào:
- 3 kênh RGB;
- 1 kênh conditioning theo loài;
- 1 kênh cạnh Sobel.

Luồng kích thước:

`64×64×5 → 32×32×64 → 16×16×128 → 8×8×256 → logit`

Mạng dùng LeakyReLU; Dropout ở hai block convolution đầu; Batch Normalization sau convolution thứ hai và thứ ba.


In [ ]:
def build_discriminator(name="discriminator"):
    image_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
    label_input = layers.Input(shape=(), dtype="int32", name="label")

    species_ch = layers.Lambda(lambda l: species_channel(l, size=IMAGE_SIZE), name="species_channel")(label_input)
    aux_ch = layers.Lambda(sobel_channel, name="sobel_channel")(image_input)
    x = layers.Concatenate(axis=-1)([image_input, species_ch, aux_ch])  # 5 channels

    x = layers.Conv2D(64, 4, strides=2, padding="same")(x)  # 32x32
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 4, strides=2, padding="same")(x)  # 16x16
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, 4, strides=2, padding="same")(x)  # 8x8
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Flatten()(x)
    output = layers.Dense(1, name="logit")(x)

    return Model([image_input, label_input], output, name=name)


_d = build_discriminator()
_d.summary()
print(f"\nTotal Discriminator parameters: {_d.count_params():,}")
del _d

## 9. Loss Functions and Optimizers

### English
Both networks use binary cross-entropy with logits.

- **Discriminator:** classify real images as `0.9` and fake images as `0`.
- **Generator:** make fake images receive target `1`.
- **Optimizer:** Adam with stage-specific learning rates and `beta_1 = 0.5`.

Real-label smoothing (`0.9`) reduces overconfident real predictions by D.

### Tiếng Việt
Cả hai mạng dùng binary cross-entropy với logits.

- **Discriminator:** phân loại ảnh thật về `0.9`, ảnh giả về `0`.
- **Generator:** cố làm ảnh giả nhận target `1`.
- **Optimizer:** Adam với learning rate riêng theo từng stage và `beta_1 = 0.5`.

Real-label smoothing (`0.9`) giúp giảm việc D quá tự tin với ảnh thật.


In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)


def discriminator_loss(real_logits, fake_logits):
    real_loss = bce(tf.ones_like(real_logits) * REAL_LABEL_SMOOTHING, real_logits)
    fake_loss = bce(tf.zeros_like(fake_logits), fake_logits)
    return real_loss + fake_loss


def generator_loss(fake_logits):
    return bce(tf.ones_like(fake_logits), fake_logits)


def make_optimizers(gen_lr, disc_lr):
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=gen_lr, beta_1=BETA_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=disc_lr, beta_1=BETA_1)
    return gen_optimizer, disc_optimizer


print("Loss functions and optimizer factory defined.")

## 10. Training Stabilizers

### English
The later stages use four stabilization methods:

- **DiffAugment translation:** randomly shifts real and fake images before D, reducing memorization.
- **Instance noise:** adds Gaussian noise to D inputs; Stage 2 starts at `0.05` and anneals it to zero.
- **EMA:** maintains a smoothed copy of Generator weights: `EMA = decay·EMA + (1-decay)·G`.
- **LR schedule:** keeps LR constant initially, then linearly reduces it to the configured floor.

These operations affect training inputs or model weights; saved/generated images themselves are not permanently augmented.

### Tiếng Việt
Các stage sau dùng bốn cơ chế ổn định:

- **DiffAugment translation:** dịch ngẫu nhiên ảnh thật và giả trước khi đưa vào D, giảm khả năng D học thuộc dữ liệu.
- **Instance noise:** thêm Gaussian noise vào input của D; Stage 2 bắt đầu ở `0.05` rồi giảm về 0.
- **EMA:** duy trì bản Generator có trọng số được làm mượt: `EMA = decay·EMA + (1-decay)·G`.
- **Lịch LR:** giữ LR cố định ở đầu stage rồi giảm tuyến tính đến mức sàn đã cấu hình.

Các phép này tác động vào quá trình train; ảnh được lưu hoặc hiển thị không bị augment vĩnh viễn.


In [ ]:
def rand_translation(x, ratio=0.125):
    """DiffAugment translation: per-sample shift up to +/- ratio of the image size, zero-filled."""
    batch_size = tf.shape(x)[0]
    image_size = tf.shape(x)[1:3]
    shift = tf.cast(tf.cast(image_size, tf.float32) * ratio + 0.5, tf.int32)
    translation_x = tf.random.uniform([batch_size, 1], -shift[0], shift[0] + 1, dtype=tf.int32)
    translation_y = tf.random.uniform([batch_size, 1], -shift[1], shift[1] + 1, dtype=tf.int32)
    grid_x = tf.clip_by_value(
        tf.expand_dims(tf.range(image_size[0]), 0) + translation_x + 1, 0, image_size[0] + 1)
    grid_y = tf.clip_by_value(
        tf.expand_dims(tf.range(image_size[1]), 0) + translation_y + 1, 0, image_size[1] + 1)
    x = tf.pad(x, [[0, 0], [1, 1], [0, 0], [0, 0]])
    x = tf.gather(x, grid_x, batch_dims=1, axis=1)
    x = tf.pad(x, [[0, 0], [0, 0], [1, 1], [0, 0]])
    x = tf.gather(x, grid_y, batch_dims=1, axis=2)
    return x


def diff_augment(x, policy=""):
    """Apply the requested DiffAugment policy. Called on real and fake alike, never on the Generator output
    that gets saved or displayed."""
    if not policy:
        return x
    for op in policy.split(","):
        op = op.strip()
        if op == "translation":
            x = rand_translation(x)
    return x


def add_instance_noise(x, std_var):
    return x + tf.random.normal(tf.shape(x)) * std_var


def update_ema(ema_model, source_model, decay):
    """ema = ema*decay + source*(1-decay), in place."""
    ema_weights = ema_model.get_weights()
    source_weights = source_model.get_weights()
    ema_model.set_weights([e * decay + s * (1.0 - decay) for e, s in zip(ema_weights, source_weights)])


def set_lr(optimizer, value):
    try:
        optimizer.learning_rate.assign(value)
    except AttributeError:
        optimizer.learning_rate = value


def lr_factor(epoch, total_epochs, decay_from, floor):
    """1.0 until decay_from (a fraction of the stage), then linear down to `floor` at the last epoch."""
    start = decay_from * total_epochs
    if epoch <= start or decay_from >= 1.0:
        return 1.0
    span = max(1.0, total_epochs - start)
    return float(1.0 - min(1.0, (epoch - start) / span) * (1.0 - floor))


def instance_noise_at(epoch, total_epochs, initial_std, anneal_frac=0.6):
    """Linear anneal from initial_std down to 0 over the first `anneal_frac` of the stage."""
    if initial_std <= 0.0:
        return 0.0
    end = max(1.0, anneal_frac * total_epochs)
    return float(initial_std * max(0.0, 1.0 - (epoch - 1) / end))


print("Stabilizers defined (DiffAugment translation, instance noise, EMA, LR schedule).")

## 11. Training Step

### English
Each batch performs the standard GAN update:

1. Sample noise and species labels.
2. Generate fake images.
3. Apply configured instance noise/DiffAugment before D.
4. Compute D loss from real and fake logits and update D.
5. Re-evaluate generated samples for G, compute G loss, and update G.
6. In Stage 3, add the mode-seeking term.

The Stage 3 mode-seeking term compares pairs with the **same class** and encourages different latent vectors to produce different images, helping resist mode collapse.

### Tiếng Việt
Mỗi batch thực hiện một bước train GAN:

1. Lấy noise và nhãn loài.
2. Generator tạo ảnh giả.
3. Áp dụng instance noise/DiffAugment theo cấu hình trước khi đưa vào D.
4. Tính loss thật/giả và cập nhật D.
5. Đánh giá lại ảnh sinh cho G, tính G loss và cập nhật G.
6. Ở Stage 3, cộng thêm mode-seeking term.

Mode-seeking của Stage 3 so sánh các cặp **cùng lớp** và khuyến khích latent vector khác nhau tạo ra ảnh khác nhau, giúp chống mode collapse.


In [ ]:
def make_train_step(generator, discriminator, gen_optimizer, disc_optimizer,
                    diffaug_policy="", ms_weight=0.0, noise_std_var=None, use_instance_noise=False):
    if noise_std_var is None:
        noise_std_var = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    paired_labels = ms_weight > 0.0

    def prepare_for_disc(images):
        """Only the copy handed to D is perturbed; the Generator's actual output stays untouched.
        Both branches are decided when the graph is built, so a stage with no noise and no DiffAugment
        (stage 1) runs exactly the plain recipe with no extra work per step."""
        if use_instance_noise:
            images = add_instance_noise(images, noise_std_var)
        return diff_augment(images, diffaug_policy)

    @tf.function
    def train_step(real_images, real_labels):
        batch_size = tf.shape(real_images)[0]
        half = batch_size // 2
        fake_bs = 2 * half

        noise = tf.random.normal([fake_bs, NOISE_DIM])
        if paired_labels:
            # sample i and sample i+half share a label, so the mode-seeking ratio measures
            # "different z, same class -> different face" and nothing else
            half_labels = tf.random.uniform([half], minval=0, maxval=NUM_CLASSES, dtype=tf.int32)
            fake_labels = tf.concat([half_labels, half_labels], axis=0)
        else:
            fake_labels = tf.random.uniform([fake_bs], minval=0, maxval=NUM_CLASSES, dtype=tf.int32)

        with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
            fake_images = generator([noise, fake_labels], training=True)

            real_in = prepare_for_disc(real_images)
            fake_in = prepare_for_disc(fake_images)

            real_logits = discriminator([real_in, real_labels], training=True)
            fake_logits = discriminator([fake_in, fake_labels], training=True)

            disc_loss = discriminator_loss(real_logits, fake_logits)
            adv_loss = generator_loss(fake_logits)

            image_gap = tf.reduce_mean(tf.abs(fake_images[:half] - fake_images[half:]), axis=[1, 2, 3])
            noise_gap = tf.reduce_mean(tf.abs(noise[:half] - noise[half:]), axis=1)
            ms_signal = tf.reduce_mean(image_gap / (noise_gap + 1e-6))

            gen_loss = adv_loss - ms_weight * ms_signal

        gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
        disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
        gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
        disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

        real_acc = tf.reduce_mean(tf.cast(tf.sigmoid(real_logits) > 0.5, tf.float32))
        fake_acc = tf.reduce_mean(tf.cast(tf.sigmoid(fake_logits) < 0.5, tf.float32))
        disc_acc = (real_acc + fake_acc) / 2.0

        return adv_loss, disc_loss, disc_acc, ms_signal

    return train_step


print("make_train_step() defined.")

## 12. Monitoring and Sample Metrics

### English
At evaluation/checkpoint points, generate fixed sample grids and compare generated images with real-image references.

Main metrics:
- `struct`: structural/shape diversity.
- `color`: color spread.
- `sharp`: edge strength/sharpness.
- `disc_acc`: Discriminator real/fake classification accuracy.
- `health`: diversity relative to the real reference.
- `score`: combined model-selection score used for the `best/` checkpoint.

Metrics are monitoring signals, not replacements for visual inspection of the generated grids.

### Tiếng Việt
Tại các mốc đánh giá/checkpoint, notebook tạo grid ảnh cố định và so sánh ảnh sinh với tham chiếu từ ảnh thật.

Các chỉ số chính:
- `struct`: độ đa dạng cấu trúc/hình dạng.
- `color`: độ đa dạng màu.
- `sharp`: độ mạnh cạnh/độ nét.
- `disc_acc`: độ chính xác thật/giả của Discriminator.
- `health`: độ đa dạng so với tham chiếu ảnh thật.
- `score`: điểm tổng hợp dùng để chọn checkpoint `best/`.

Các metric dùng để theo dõi; vẫn cần xem trực tiếp grid ảnh sinh.


In [ ]:
SAMPLES_PER_CLASS = 16
GRID_COLS = 8
ROWS_PER_CLASS = SAMPLES_PER_CLASS // GRID_COLS
GRID_ROWS = ROWS_PER_CLASS * NUM_CLASSES
FIG_W, FIG_H, FIG_DPI = 16.0, 9.0, 120  # -> 1920 x 1080 px, one 1080p video frame

FIXED_NOISE = tf.random.normal([SAMPLES_PER_CLASS * NUM_CLASSES, NOISE_DIM], seed=RANDOM_SEED)
FIXED_LABELS = tf.constant(
    [label for label in sorted(LABEL_TO_SPECIES) for _ in range(SAMPLES_PER_CLASS)], dtype=tf.int32)


def _grid_axes_rects(n_rows, n_cols, fig_w, fig_h, block_rows):
    """Exact axes rectangles (figure fractions) for a centered grid of square cells, plus the vertical
    center of each class block for its left-margin label. Hand-placed, so nothing can overlap."""
    left_margin, right_margin = 0.050, 0.012
    top_margin, bottom_margin = 0.085, 0.022
    gap = 0.055        # inches between cells
    block_gap = 0.28   # extra inches between the dog block and the cat block

    avail_w = fig_w * (1.0 - left_margin - right_margin)
    avail_h = fig_h * (1.0 - top_margin - bottom_margin) - block_gap
    cell = min((avail_w - (n_cols - 1) * gap) / n_cols,
               (avail_h - (n_rows - 1) * gap) / n_rows)

    grid_w = n_cols * cell + (n_cols - 1) * gap
    grid_h = n_rows * cell + (n_rows - 1) * gap + block_gap
    x0 = fig_w * left_margin + (avail_w - grid_w) / 2.0
    band_h = fig_h * (1.0 - top_margin - bottom_margin)
    y_top = fig_h * (1.0 - top_margin) - (band_h - grid_h) / 2.0

    rects, blocks = [], []
    for r in range(n_rows):
        extra = block_gap if r >= block_rows else 0.0
        y = y_top - (r + 1) * cell - r * gap - extra
        for c in range(n_cols):
            rects.append(((x0 + c * (cell + gap)) / fig_w, y / fig_h, cell / fig_w, cell / fig_h))
    for b in range(n_rows // block_rows):
        first, last = b * block_rows * n_cols, (b + 1) * block_rows * n_cols - 1
        blocks.append((0.5 * ((rects[first][1] + rects[first][3]) + rects[last][1]), x0 / fig_w))
    return rects, blocks


def save_sample_grid(model, epoch, save_dir, stage_label, model_label="G"):
    generated = model([FIXED_NOISE, FIXED_LABELS], training=False)
    generated = ((generated.numpy() + 1.0) * 127.5).clip(0, 255).astype("uint8")

    fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=FIG_DPI, facecolor="white")
    rects, blocks = _grid_axes_rects(GRID_ROWS, GRID_COLS, FIG_W, FIG_H, ROWS_PER_CLASS)
    for i, rect in enumerate(rects):
        ax = fig.add_axes(rect)
        ax.imshow(generated[i])
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    for b, (y_center, x_left) in enumerate(blocks):
        fig.text(x_left - 0.016, y_center, LABEL_TO_SPECIES[b].upper(), rotation=90,
                 va="center", ha="center", fontsize=15, fontweight="bold", color="#444444")
    fig.text(0.5, 0.955, f"{stage_label}  --  epoch {epoch}   ({model_label})",
             ha="center", va="center", fontsize=17, fontweight="bold")

    out_path = save_dir / f"epoch_{epoch:03d}.png"
    fig.savefig(out_path, dpi=FIG_DPI, facecolor="white")
    plt.close(fig)
    return out_path


print(f"Sample grid: {SAMPLES_PER_CLASS} per class, {GRID_COLS}x{GRID_ROWS}, "
      f"{int(FIG_W * FIG_DPI)}x{int(FIG_H * FIG_DPI)} px.")

In [ ]:
EVAL_SAMPLES_PER_CLASS = 64
EVAL_NOISE = tf.random.normal([EVAL_SAMPLES_PER_CLASS, NOISE_DIM], seed=RANDOM_SEED + 1)


def structure_diversity(images, size=32):
    """1 - mean pairwise correlation of grayscale, downsampled, per-image standardized images.
    ~0.0 = every sample has the same shape (mode collapse); higher = genuinely different shapes.
    Brightness and colour are normalized away, so this measures SHAPE only."""
    gray = tf.image.rgb_to_grayscale(images)
    gray = tf.image.resize(gray, [size, size], method="area")
    flat = tf.reshape(gray, [tf.shape(gray)[0], -1])
    flat = (flat - tf.reduce_mean(flat, axis=1, keepdims=True)) / (
        tf.math.reduce_std(flat, axis=1, keepdims=True) + 1e-6)
    n = tf.cast(tf.shape(flat)[0], tf.float32)
    d = tf.cast(tf.shape(flat)[1], tf.float32)
    gram = tf.matmul(flat, flat, transpose_b=True) / d
    off_diagonal = tf.reduce_sum(gram) - tf.linalg.trace(gram)
    return float(1.0 - off_diagonal / (n * (n - 1.0)))


def color_diversity(images):
    """Spread of per-image mean RGB across the batch. Low = one narrow palette."""
    per_image = tf.reduce_mean(images, axis=[1, 2])
    return float(tf.reduce_mean(tf.math.reduce_std(per_image, axis=0)))


def sharpness(images):
    """Mean Sobel gradient magnitude (unnormalized). Low = blurry."""
    gray = tf.image.rgb_to_grayscale((images + 1.0) / 2.0)
    edges = tf.image.sobel_edges(gray)
    return float(tf.reduce_mean(tf.sqrt(tf.reduce_sum(tf.square(edges), axis=-1) + 1e-8)))


def measure_images(images):
    return {"struct": structure_diversity(images),
            "color": color_diversity(images),
            "sharp": sharpness(images)}


def evaluate_generator(model):
    """Per-class metrics on the fixed evaluation noise."""
    out = {}
    for label, species in LABEL_TO_SPECIES.items():
        labels = tf.fill([EVAL_SAMPLES_PER_CLASS], label)
        images = model([EVAL_NOISE, labels], training=False)
        for key, value in measure_images(images).items():
            out[f"{key}_{species}"] = value
    return out


def _real_reference():
    refs = {}
    for species, label in SPECIES_TO_LABEL.items():
        subset = train_manifest[train_manifest["species"] == species].head(EVAL_SAMPLES_PER_CLASS * 2)
        ds = build_dataset(subset, LOCAL_IMAGES_DIR, EVAL_SAMPLES_PER_CLASS,
                           shuffle=False, augment_flag=False)
        images, _ = next(iter(ds))
        for key, value in measure_images(images).items():
            refs[f"{key}_{species}"] = value
    return refs


REAL_REFS = _real_reference()
print("Real-image references (the numbers every stage is compared against):")
for species in SPECIES_TO_LABEL:
    print(f"  {species:4s}  struct={REAL_REFS[f'struct_{species}']:.3f}  "
          f"color={REAL_REFS[f'color_{species}']:.3f}  sharp={REAL_REFS[f'sharp_{species}']:.4f}")

## 13. Checkpoints, Best Model, and Collapse Guard

### English
Each stage stores:
- **rolling checkpoint:** latest G/D weights for resume;
- **snapshots:** permanent periodic copies for rollback;
- **best checkpoint:** Generator with the highest recorded selection score;
- **history.csv:** per-epoch training metrics.

Stage 3 stops early if `health < 0.60` for `2` consecutive checks. This guard is based on generated-image diversity relative to the real reference.

### Tiếng Việt
Mỗi stage lưu:
- **rolling checkpoint:** weight G/D mới nhất để resume;
- **snapshots:** các bản định kỳ không ghi đè để có thể quay lại;
- **best checkpoint:** Generator có selection score cao nhất;
- **history.csv:** metric theo từng epoch.

Stage 3 early-stop nếu `health < 0.60` trong `2` lần kiểm tra liên tiếp. Điều kiện này dựa trên độ đa dạng ảnh sinh so với tham chiếu ảnh thật.


- **web progression:** selected Generator/EMA weights for the later browser animation; these files are separate from training-resume checkpoints.

- **web progression:** các Generator/EMA weight được chọn riêng cho animation trên web; chúng tách biệt với checkpoint dùng để resume training.


In [ ]:
import json
import shutil


def quality_score(metrics, refs=None):
    refs = refs or REAL_REFS
    gen_sharp = sum(metrics[f"sharp_{s}"] for s in SPECIES_TO_LABEL)
    real_sharp = sum(refs[f"sharp_{s}"] for s in SPECIES_TO_LABEL)
    sharp_ratio = min(1.0, gen_sharp / (real_sharp + 1e-8))
    health = min(metrics[f"struct_{s}"] / (refs[f"struct_{s}"] + 1e-8) for s in SPECIES_TO_LABEL)
    return float(sharp_ratio * min(1.0, health / HEALTH_TARGET)), float(health)


def save_rolling(model_dir, generator, discriminator, generator_ema=None):
    generator.save_weights(model_dir / "generator.weights.h5")
    discriminator.save_weights(model_dir / "discriminator.weights.h5")
    if generator_ema is not None:
        generator_ema.save_weights(model_dir / "generator_ema.weights.h5")


def save_snapshot(snap_dir, epoch, generator, generator_ema=None):
    generator.save_weights(snap_dir / f"generator_e{epoch:03d}.weights.h5")
    if generator_ema is not None:
        generator_ema.save_weights(snap_dir / f"generator_ema_e{epoch:03d}.weights.h5")


def save_best(best_dir, epoch, score, health, metrics, showcase_model):
    showcase_model.save_weights(best_dir / "generator_best.weights.h5")
    info = {"epoch": int(epoch), "score": round(float(score), 4), "health": round(float(health), 4)}
    info.update({k: round(float(v), 4) for k, v in metrics.items()})
    (best_dir / "best_info.json").write_text(json.dumps(info, indent=2))
    return info


def load_best_info(best_dir):
    path = best_dir / "best_info.json"
    if path.exists():
        try:
            return json.loads(path.read_text())
        except json.JSONDecodeError:
            return None
    return None


def _read_web_progression_manifest():
    WEB_PROGRESSION_DIR.mkdir(parents=True, exist_ok=True)
    if WEB_PROGRESSION_MANIFEST.exists():
        try:
            return json.loads(WEB_PROGRESSION_MANIFEST.read_text())
        except (json.JSONDecodeError, OSError):
            pass
    return {
        "model": MODEL_SUBDIR,
        "image_size": int(IMAGE_SIZE),
        "noise_dim": int(NOISE_DIM),
        "embedding_dim": int(EMBEDDING_DIM),
        "species_to_label": {str(k): int(v) for k, v in SPECIES_TO_LABEL.items()},
        "note": "Use the same latent z and species label across checkpoints for the web progression.",
        "checkpoints": [],
    }


def _write_web_progression_manifest(data):
    WEB_PROGRESSION_DIR.mkdir(parents=True, exist_ok=True)
    stage_order = {"stage1": 1, "stage2": 2, "stage3": 3}
    data["checkpoints"] = sorted(
        data.get("checkpoints", []),
        key=lambda x: (
            99 if x.get("kind") == "final_best" else stage_order.get(x.get("stage"), 98),
            int(x.get("epoch", 10**9)),
        ),
    )
    WEB_PROGRESSION_MANIFEST.write_text(json.dumps(data, indent=2))


def save_web_progression(stage_key, epoch, showcase_model, model_kind):
    """Save a web-only Generator checkpoint and upsert its manifest entry."""
    WEB_PROGRESSION_DIR.mkdir(parents=True, exist_ok=True)
    kind_slug = "ema" if str(model_kind).upper() == "EMA" else "generator"
    filename = f"{stage_key}_e{int(epoch):03d}_{kind_slug}.weights.h5"
    path = WEB_PROGRESSION_DIR / filename
    showcase_model.save_weights(path)

    data = _read_web_progression_manifest()
    entry = {
        "stage": stage_key,
        "epoch": int(epoch),
        "kind": "progression",
        "model_kind": str(model_kind),
        "file": filename,
    }
    checkpoints = [
        x for x in data.get("checkpoints", [])
        if not (x.get("stage") == stage_key and int(x.get("epoch", -1)) == int(epoch)
                and x.get("kind") == "progression")
    ]
    checkpoints.append(entry)
    data["checkpoints"] = checkpoints
    _write_web_progression_manifest(data)
    return path


def save_web_final_best(best_dir, best_epoch, best_score):
    """Copy the best Stage 3 Generator into the web-progression package."""
    src = best_dir / "generator_best.weights.h5"
    if not src.exists():
        return None

    WEB_PROGRESSION_DIR.mkdir(parents=True, exist_ok=True)
    filename = "generator_final_best.weights.h5"
    dst = WEB_PROGRESSION_DIR / filename
    shutil.copy2(src, dst)

    data = _read_web_progression_manifest()
    checkpoints = [x for x in data.get("checkpoints", []) if x.get("kind") != "final_best"]
    checkpoints.append({
        "stage": "stage3",
        "epoch": int(best_epoch),
        "kind": "final_best",
        "model_kind": "EMA",
        "score": float(best_score),
        "file": filename,
    })
    data["checkpoints"] = checkpoints
    _write_web_progression_manifest(data)
    return dst


print("Checkpoint helpers defined.")

## 14. Stage Runner

### English
`run_stage()` executes one complete stage. It:
1. creates G, D, optimizers, and optional EMA;
2. resumes the same stage if checkpoints exist;
3. otherwise loads weights from the previous stage;
4. applies the stage-specific LR/noise schedule;
5. trains epoch by epoch;
6. evaluates, logs, saves checkpoints/snapshots, and updates `best/`;
7. applies the Stage 3 collapse early-stop rule.

Stage 2 loads both G and D from Stage 1. Stage 3 loads the Stage 2 EMA Generator and the Stage 2 Discriminator.

### Tiếng Việt
`run_stage()` thực hiện toàn bộ một stage:
1. tạo G, D, optimizer và EMA nếu cần;
2. resume chính stage đó nếu đã có checkpoint;
3. nếu chưa có thì nạp weight từ stage trước;
4. áp dụng lịch LR/noise của stage;
5. train theo từng epoch;
6. đánh giá, ghi log, lưu checkpoint/snapshot và cập nhật `best/`;
7. áp dụng early-stop chống collapse ở Stage 3.

Stage 2 nạp cả G và D từ Stage 1. Stage 3 nạp EMA Generator của Stage 2 và Discriminator của Stage 2.


In [ ]:
import time


def run_stage(stage_key, epochs=None):
    cfg = dict(STAGES[stage_key])
    total_epochs = int(epochs or cfg["epochs"])

    model_dir = MODELS_DIR / MODEL_SUBDIR / stage_key
    figures_dir = FIGURES_DIR / MODEL_SUBDIR / stage_key
    snap_dir = model_dir / "snapshots"
    best_dir = model_dir / "best"
    for directory in (model_dir, figures_dir, snap_dir, best_dir, WEB_PROGRESSION_DIR):
        directory.mkdir(parents=True, exist_ok=True)

    history_path = model_dir / "history.csv"
    gen_path = model_dir / "generator.weights.h5"
    disc_path = model_dir / "discriminator.weights.h5"
    ema_path = model_dir / "generator_ema.weights.h5"

    use_ema = cfg["ema_decay"] > 0.0
    generator = build_generator()
    discriminator = build_discriminator(name=f"discriminator_{stage_key}")
    generator_ema = build_generator(name="generator_ema") if use_ema else None
    gen_optimizer, disc_optimizer = make_optimizers(cfg["gen_lr"], cfg["disc_lr"])
    noise_std_var = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    train_step = make_train_step(generator, discriminator, gen_optimizer, disc_optimizer,
                                 diffaug_policy=cfg["diffaug"], ms_weight=cfg["ms_weight"],
                                 noise_std_var=noise_std_var,
                                 use_instance_noise=cfg["instance_noise"] > 0.0)

    # ---- where do the starting weights come from? ----
    start_epoch, history = 1, []
    if history_path.exists() and gen_path.exists() and disc_path.exists():
        history_df = pd.read_csv(history_path)
        history = history_df.to_dict("records")
        start_epoch = int(history_df["epoch"].max()) + 1
        generator.load_weights(gen_path)
        discriminator.load_weights(disc_path)
        if use_ema:
            if ema_path.exists():
                generator_ema.load_weights(ema_path)
            else:
                generator_ema.set_weights(generator.get_weights())
        print(f"[{stage_key}] Resuming from epoch {start_epoch} (found this stage's checkpoint).")
    elif cfg["init_from"]:
        prev_dir = MODELS_DIR / MODEL_SUBDIR / cfg["init_from"]
        prev_ema = prev_dir / "generator_ema.weights.h5"
        prev_gen = prev_ema if (cfg["init_from_ema"] and prev_ema.exists()) else prev_dir / "generator.weights.h5"
        prev_disc = prev_dir / "discriminator.weights.h5"
        assert prev_gen.exists() and prev_disc.exists(), (
            f"[{stage_key}] needs {cfg['init_from']} to be trained first -- "
            f"missing {prev_gen if not prev_gen.exists() else prev_disc}")
        generator.load_weights(prev_gen)
        discriminator.load_weights(prev_disc)
        if use_ema:
            generator_ema.set_weights(generator.get_weights())
        print(f"[{stage_key}] Starting from {cfg['init_from']} weights ({prev_gen.name} + {prev_disc.name}).")
    else:
        print(f"[{stage_key}] Starting fresh (random initialization).")
        if use_ema:
            generator_ema.set_weights(generator.get_weights())

    if start_epoch > total_epochs:
        print(f"[{stage_key}] Already at epoch {start_epoch - 1}/{total_epochs} -- nothing to do.")
        return generator_ema if use_ema else generator, discriminator, pd.DataFrame(history)

    best_info = load_best_info(best_dir)
    best_score = float(best_info["score"]) if best_info else -1.0
    best_epoch = int(best_info["epoch"]) if best_info else -1
    collapse_hits = 0
    showcase = generator_ema if use_ema else generator
    showcase_label = "EMA" if use_ema else "G"

    print(f"[{stage_key}] {total_epochs} epochs | G lr={cfg['gen_lr']:.1e} D lr={cfg['disc_lr']:.1e} | "
          f"noise={cfg['instance_noise']} diffaug='{cfg['diffaug']}' ema={cfg['ema_decay']} "
          f"ms={cfg['ms_weight']} lr_floor={cfg['lr_floor']} | sampling the {showcase_label} model")

    for epoch in range(start_epoch, total_epochs + 1):
        epoch_start = time.time()
        factor = lr_factor(epoch, total_epochs, cfg["lr_decay_from"], cfg["lr_floor"])
        set_lr(gen_optimizer, cfg["gen_lr"] * factor)
        set_lr(disc_optimizer, cfg["disc_lr"] * factor)
        noise_std_var.assign(instance_noise_at(epoch, total_epochs, cfg["instance_noise"]))

        gen_losses, disc_losses, disc_accs, ms_values = [], [], [], []
        for real_images, real_labels in train_ds:
            gen_loss, disc_loss, disc_acc, ms_signal = train_step(real_images, real_labels)
            gen_losses.append(float(gen_loss))
            disc_losses.append(float(disc_loss))
            disc_accs.append(float(disc_acc))
            ms_values.append(float(ms_signal))

        if use_ema:
            update_ema(generator_ema, generator, cfg["ema_decay"])

        metrics = evaluate_generator(showcase)
        score, health = quality_score(metrics)
        epoch_time = time.time() - epoch_start
        sharp_ratio = (sum(metrics[f"sharp_{s}"] for s in SPECIES_TO_LABEL)
                       / sum(REAL_REFS[f"sharp_{s}"] for s in SPECIES_TO_LABEL))

        row = {"stage": stage_key, "epoch": epoch,
               "gen_loss": float(np.mean(gen_losses)), "disc_loss": float(np.mean(disc_losses)),
               "disc_acc": float(np.mean(disc_accs)), "ms_signal": float(np.mean(ms_values)),
               "score": score, "health": health, "sharp_ratio": float(sharp_ratio),
               "lr_factor": factor, "instance_noise": float(noise_std_var.numpy()),
               "seconds": epoch_time}
        row.update({k: float(v) for k, v in metrics.items()})
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        print(f"[{stage_key}] epoch {epoch}/{total_epochs} | "
              f"gen={row['gen_loss']:.3f} disc={row['disc_loss']:.3f} acc={row['disc_acc']:.3f} "
              f"ms={row['ms_signal']:.3f} | struct d/c {metrics['struct_dog']:.3f}/{metrics['struct_cat']:.3f} "
              f"(health {health:.2f}) color d/c {metrics['color_dog']:.3f}/{metrics['color_cat']:.3f} | "
              f"sharp {sharp_ratio:.2f}x | score {score:.3f} "
              f"(best {'--' if best_epoch < 0 else f'{best_score:.3f} @ e{best_epoch}'}) | {epoch_time:.1f}s")

        if row["disc_acc"] > 0.95:
            print("  WARNING: Discriminator accuracy above 0.95 -- it may be overpowering the Generator.")
        elif row["disc_acc"] < 0.55:
            print("  WARNING: Discriminator accuracy below 0.55 -- it may have collapsed.")
        for species in SPECIES_TO_LABEL:
            ratio = metrics[f"struct_{species}"] / (REAL_REFS[f"struct_{species}"] + 1e-8)
            if ratio < COLLAPSE_STOP_RATIO:
                print(f"  WARNING: {species} shape diversity is {ratio:.2f}x the real reference -- "
                      f"the {species} samples may be converging on one face. Open the grid and check.")

        if epoch % CHECKPOINT_EVERY == 0 or epoch == total_epochs:
            save_rolling(model_dir, generator, discriminator, generator_ema)
            grid_path = save_sample_grid(showcase, epoch, figures_dir,
                                         f"{MODEL_SUBDIR} {stage_key}", showcase_label)
            print(f"  Checkpoint saved. Sample grid: {grid_path} -- look at it before trusting any number above.")
            if score > best_score:
                best_score, best_epoch = score, epoch
                save_best(best_dir, epoch, score, health, metrics, showcase)
                print(f"  New best model for this stage (score {score:.3f}) -> {best_dir/'generator_best.weights.h5'}")

        if epoch in WEB_PROGRESSION_EPOCHS.get(stage_key, []):
            web_path = save_web_progression(stage_key, epoch, showcase, showcase_label)
            print(f"  Web progression Generator kept: {web_path}")

        if epoch % cfg["snapshot_every"] == 0:
            save_snapshot(snap_dir, epoch, generator, generator_ema)
            print(f"  Permanent snapshot kept: {snap_dir}/generator_e{epoch:03d}.weights.h5")

        if cfg["early_stop"]:
            collapse_hits = collapse_hits + 1 if health < COLLAPSE_STOP_RATIO else 0
            if collapse_hits >= COLLAPSE_PATIENCE:
                save_rolling(model_dir, generator, discriminator, generator_ema)
                save_snapshot(snap_dir, epoch, generator, generator_ema)
                print(f"\n[{stage_key}] EARLY STOP at epoch {epoch}: shape diversity stayed below "
                      f"{COLLAPSE_STOP_RATIO:.2f}x the real reference for {COLLAPSE_PATIENCE} checks in a row.")
                print(f"[{stage_key}] Use the best model instead: {best_dir/'generator_best.weights.h5'} "
                      f"(epoch {best_epoch}, score {best_score:.3f}).")
                break

    if stage_key == "stage3":
        final_web_path = save_web_final_best(best_dir, best_epoch, best_score)
        if final_web_path is not None:
            print(f"[stage3] Final best Generator copied for web progression: {final_web_path}")
            print(f"[stage3] Web progression manifest: {WEB_PROGRESSION_MANIFEST}")
        else:
            print("[stage3] WARNING: no best Generator file was available to copy into web_progression/.")

    print(f"\n[{stage_key}] Done. Best model: epoch {best_epoch}, score {best_score:.3f} -> "
          f"{best_dir/'generator_best.weights.h5'}")
    return showcase, discriminator, pd.DataFrame(history)


print("run_stage() defined.")

## 15. Stage 1 — Learn Basic Structure

### English
**150 epochs**

Parameters:
- G LR = `2e-4`
- D LR = `2e-4`
- instance noise = off
- DiffAugment = off
- EMA = off
- mode-seeking = off
- LR decay = off
- snapshot every 25 epochs

This stage starts from random weights. Its purpose is to establish the basic dog/cat structure before adding stronger stabilization methods.

### Tiếng Việt
**150 epoch**

Tham số:
- LR G = `2e-4`
- LR D = `2e-4`
- instance noise = tắt
- DiffAugment = tắt
- EMA = tắt
- mode-seeking = tắt
- giảm LR = tắt
- snapshot mỗi 25 epoch

Stage này bắt đầu từ trọng số ngẫu nhiên. Mục tiêu là học cấu trúc cơ bản của chó/mèo trước khi thêm các cơ chế ổn định mạnh hơn.


In [ ]:
gen_s1, disc_s1, hist_s1 = run_stage("stage1")

## 16. Stage 2 — Controlled Refinement

### English
**200 epochs**

Parameters:
- G LR = `1.5e-4`
- D LR = `5e-5`
- instance noise = `0.05 → 0` during the first 60%
- DiffAugment = translation
- EMA decay = `0.95`
- mode-seeking = off
- LR decay starts at 70%, floor = `0.4×` base LR
- snapshot every 25 epochs

D uses a lower LR than G (**TTUR**) to reduce the chance that D learns much faster than G. EMA provides a smoother Generator copy for monitoring and later initialization.

### Tiếng Việt
**200 epoch**

Tham số:
- LR G = `1.5e-4`
- LR D = `5e-5`
- instance noise = `0.05 → 0` trong 60% đầu
- DiffAugment = translation
- EMA decay = `0.95`
- mode-seeking = tắt
- bắt đầu giảm LR ở 70%, sàn = `0.4×` LR gốc
- snapshot mỗi 25 epoch

D dùng LR thấp hơn G (**TTUR**) để giảm nguy cơ D học nhanh hơn quá nhiều. EMA tạo một bản Generator ổn định hơn để theo dõi và dùng làm điểm bắt đầu cho stage sau.


In [ ]:
gen_s2, disc_s2, hist_s2 = run_stage("stage2")

## 17. Stage 3 — Sharpen and Preserve Diversity

### English
**Up to 100 epochs**

Parameters:
- G LR = `5e-5`
- D LR = `2.5e-5`
- instance noise = off
- DiffAugment = translation
- EMA decay = `0.90`
- mode-seeking weight = `0.2`
- LR decay starts at 30%, floor = `0.3×` base LR
- snapshot every 10 epochs
- collapse early stop = on

Stage 3 starts from the Stage 2 EMA Generator. Low LR is used for fine adjustment, while mode-seeking and the `health` guard help preserve diversity. Prefer `stage3/best/generator_best.weights.h5` for later export rather than assuming the last epoch is best.

### Tiếng Việt
**Tối đa 100 epoch**

Tham số:
- LR G = `5e-5`
- LR D = `2.5e-5`
- instance noise = tắt
- DiffAugment = translation
- EMA decay = `0.90`
- mode-seeking weight = `0.2`
- bắt đầu giảm LR ở 30%, sàn = `0.3×` LR gốc
- snapshot mỗi 10 epoch
- early stop collapse = bật

Stage 3 bắt đầu từ EMA Generator của Stage 2. LR thấp dùng để tinh chỉnh; mode-seeking và `health` guard giúp giữ độ đa dạng. Khi export nên ưu tiên `stage3/best/generator_best.weights.h5` thay vì mặc định lấy epoch cuối.


In [ ]:
gen_s3, disc_s3, hist_s3 = run_stage("stage3")

## 18. Review the Full Training Run

### English
Combine `history.csv` from all stages into one timeline and plot:
- G/D losses;
- per-class structural diversity against real-image references and the collapse threshold;
- sharpness ratio, model-selection score, and D accuracy.

Use these plots together with generated sample grids to decide whether training is improving or becoming unstable.

### Tiếng Việt
Nối `history.csv` của cả ba stage thành một timeline và vẽ:
- loss của G/D;
- độ đa dạng cấu trúc theo từng lớp so với ảnh thật và ngưỡng collapse;
- tỷ lệ độ nét, selection score và độ chính xác của D.

Dùng các biểu đồ này cùng grid ảnh sinh để đánh giá model đang cải thiện hay mất ổn định.


In [ ]:
frames = []
offset = 0
for key in STAGES:
    path = MODELS_DIR / MODEL_SUBDIR / key / "history.csv"
    if path.exists():
        df = pd.read_csv(path)
        df["global_epoch"] = df["epoch"] + offset
        offset = df["global_epoch"].max()
        frames.append(df)

if not frames:
    print("No history yet -- run at least one stage first.")
else:
    all_hist = pd.concat(frames, ignore_index=True)
    boundaries = all_hist.groupby("stage")["global_epoch"].min().sort_values().tolist()[1:]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(all_hist["global_epoch"], all_hist["gen_loss"], label="Generator")
    axes[0].plot(all_hist["global_epoch"], all_hist["disc_loss"], label="Discriminator")
    axes[0].set_title("Losses")
    axes[0].set_xlabel("epoch (all stages)")
    axes[0].legend(fontsize=8)

    for species, colour in zip(SPECIES_TO_LABEL, ["tab:blue", "tab:orange"]):
        axes[1].plot(all_hist["global_epoch"], all_hist[f"struct_{species}"], color=colour, label=species)
        axes[1].axhline(REAL_REFS[f"struct_{species}"], color=colour, alpha=0.35, lw=1)
        axes[1].axhline(REAL_REFS[f"struct_{species}"] * COLLAPSE_STOP_RATIO, color=colour,
                        alpha=0.6, lw=1, ls="--")
    axes[1].set_title("Shape diversity vs real (solid = real, dashed = collapse threshold)")
    axes[1].set_xlabel("epoch (all stages)")
    axes[1].legend(fontsize=8)

    axes[2].plot(all_hist["global_epoch"], all_hist["sharp_ratio"], label="sharpness / real")
    axes[2].plot(all_hist["global_epoch"], all_hist["score"], label="quality score")
    axes[2].plot(all_hist["global_epoch"], all_hist["disc_acc"], label="disc_acc", alpha=0.6)
    axes[2].set_title("Sharpness, score, D accuracy")
    axes[2].set_xlabel("epoch (all stages)")
    axes[2].legend(fontsize=8)

    for ax in axes:
        for boundary in boundaries:
            ax.axvline(boundary, color="grey", lw=1, ls=":")
        ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "05_all_stages.png", dpi=150)
    plt.show()

    for key in STAGES:
        info = load_best_info(MODELS_DIR / MODEL_SUBDIR / key / "best")
        if info:
            print(f"{key}: best epoch {info['epoch']}, score {info['score']}, health {info['health']}")

In [ ]:
from PIL import Image as PILImage

latest_stage = None
for key in STAGES:
    if (FIGURES_DIR / MODEL_SUBDIR / key).exists() and any((FIGURES_DIR / MODEL_SUBDIR / key).glob("epoch_*.png")):
        latest_stage = key

if latest_stage is None:
    print("No sample grids yet.")
else:
    grids = sorted((FIGURES_DIR / MODEL_SUBDIR / latest_stage).glob("epoch_*.png"))
    latest = grids[-1]
    fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(PILImage.open(latest))
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Showing {latest_stage} / {latest.name} ({len(grids)} grids saved in this stage).")

## Summary

### English
`kinkySobel` is a 64×64 conditional DCGAN trained in three sequential stages:

1. **Stage 1:** learn basic structure with equal G/D learning rates.
2. **Stage 2:** refine with TTUR, instance noise, translation DiffAugment, and EMA.
3. **Stage 3:** fine-tune with low LR, EMA, translation DiffAugment, mode-seeking loss, and collapse early stop.

The Discriminator uses RGB + species conditioning + Sobel edges. Training uses BCE, Adam (`beta_1=0.5`), one-sided real-label smoothing (`0.9`), per-epoch logging, sample grids, rolling checkpoints, permanent snapshots, and score-based best-model saving.

Recommended export weight: `models/kinkySobel/stage3/best/generator_best.weights.h5`.

### Tiếng Việt
`kinkySobel` là conditional DCGAN 64×64 được train tuần tự qua ba stage:

1. **Stage 1:** học cấu trúc cơ bản với LR G/D bằng nhau.
2. **Stage 2:** tinh chỉnh bằng TTUR, instance noise, translation DiffAugment và EMA.
3. **Stage 3:** fine-tune với LR thấp, EMA, translation DiffAugment, mode-seeking loss và early stop chống collapse.

Discriminator dùng RGB + conditioning theo loài + cạnh Sobel. Training dùng BCE, Adam (`beta_1=0.5`), real-label smoothing một phía (`0.9`), log theo epoch, grid ảnh mẫu, rolling checkpoint, snapshot vĩnh viễn và lưu best model theo score.

Weight khuyến nghị để export: `models/kinkySobel/stage3/best/generator_best.weights.h5`.


For deployment, `models/kinkySobel/web_progression/manifest.json` lists the selected progression weights. Reuse one `z` and one species label across those weights to animate the same generated subject from early training to the final best model.

Khi deploy, `models/kinkySobel/web_progression/manifest.json` liệt kê các weight progression đã chọn. Dùng cùng một `z` và cùng nhãn loài qua tất cả weight để animation thể hiện cùng một ảnh từ giai đoạn train sớm đến best model cuối.
